# 18 — Video Grounding, Summarization & Event Extraction

**Orbit:** Multimodal AI · **Difficulty:** Advanced · **Reading time:** 35 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/18_video_grounding_summarization_and_event_extraction.ipynb)

**Prerequisites:** [07 — Structured Outputs, Grounding & Localization](07_structured_outputs_grounding_and_localization.ipynb) · [09 — Captioning as a Text Bridge](09_captioning_as_a_text_bridge.ipynb) · [17 — Video Understanding, Frame Sampling & Temporal Reasoning](17_video_understanding_frame_sampling_and_temporal_reasoning.ipynb)

**What you will learn:**
- How to link natural-language queries to specific timestamps in a video (temporal grounding).
- How to generate hierarchical summaries from frame-level captions to video-level narratives.
- How to extract structured event records (JSON) from video content.
- How to build a simple clip-retrieval system over captioned video.

## Learning Objectives

By the end of this notebook you will be able to:

1. **Build event timelines** from frame-level VLM captions, detecting scene boundaries by comparing consecutive descriptions.
2. **Implement temporal grounding** — linking free-text queries to video timestamps using a caption-then-search strategy.
3. **Generate hierarchical video summaries** (frame → scene → video) that compress long footage into concise narratives.
4. **Extract structured events** (JSON with actor, action, timestamp, objects) from video content using a language model.
5. **Perform text-based clip retrieval** over a captioned video, returning ranked frame sequences for a natural-language query.
6. **Connect video understanding to captioning (notebook 09) and structured outputs (notebook 07)**, seeing how each capability feeds the next.

## Video Grounding — Linking Language to Time

Video grounding links natural-language descriptions to specific time segments in a video. Given a query like *"the moment the person picks up the red cup,"* the system must identify that this event happens between timestamps 12.5 s and 14.2 s. This is harder than image grounding because the answer lives in the **temporal** dimension rather than the spatial one.

Two main approaches exist. **(a) Caption-then-search:** caption every frame or short segment independently, then find the captions that best match the query using text similarity. This is modular — any VLM can provide captions and any text-matching technique can rank them. **(b) End-to-end:** train a specialised model (like Moment-DETR or TimeChat) to directly predict temporal boundaries from a joint video-text representation.

In this notebook we adopt approach **(a)** because it works with any off-the-shelf VLM and requires no fine-tuning. We caption sampled frames, build a searchable index, and retrieve timestamps by text matching. The trade-off is that our temporal resolution is limited by our sampling rate.

## Video Summarization — From Frames to Narrative

Video summarization compresses potentially hours of footage into key moments or a short textual narrative. Our approach is **hierarchical**: first caption individual frames, then group those captions into *scenes* (contiguous segments with consistent content), summarize each scene into a single sentence, and finally summarize across scenes to produce a video-level narrative.

This mirrors how humans naturally summarize video — we remember scenes and transitions, not individual frames. The central challenge is **redundancy**. Many consecutive frames depict the same activity; a good summarizer must distinguish what is *new* from what is *repeated*. Our scene-detection step addresses this by collapsing runs of similar captions into a single scene. The scene-level summaries then capture only the distinct events, and the video-level summary weaves them into a coherent paragraph.

## Event Extraction — Structured Records from Video

Event extraction transforms unstructured video into a **queryable database** of structured records. Each event record contains: **actor** (who), **action** (what), **start_time / end_time** (when), and the **objects** involved.

Applications include generating meeting minutes from recordings, producing surveillance logs from security footage, and creating sports highlight databases. By converting video into structured data we bridge the gap between visual media and the relational databases that power most enterprise workflows.

In [ ]:
!pip install -q transformers torch qwen-vl-utils opencv-python-headless pillow bitsandbytes accelerate

In [ ]:
import json, math, os, time, re, textwrap, pathlib
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Tuple, Optional
from collections import Counter

import cv2
import numpy as np
from PIL import Image
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

plt.rcParams.update({"font.size": 11, "figure.dpi": 110})
print(f"torch {torch.__version__}  •  CUDA {torch.cuda.is_available()}")
print("Dependencies ready.")

## Synthetic Video Generation

We create a 30-second synthetic video at 24 fps with five distinct scenes, each lasting six seconds. Every scene uses a unique background colour and overlaid text describing the action, giving us ground-truth events for evaluation.

In [ ]:
@dataclass
class GroundTruthEvent:
    event_id: int
    actor: str
    action: str
    start_time: float
    end_time: float
    objects: List[str] = field(default_factory=list)

SCENES = [
    {"colour": (0, 0, 200),   "text": "Person enters room",       "action": "enters room",       "objects": ["door"]},
    {"colour": (0, 180, 0),   "text": "Person picks up object",   "action": "picks up object",   "objects": ["cup"]},
    {"colour": (200, 80, 0),  "text": "Person reads document",    "action": "reads document",    "objects": ["document"]},
    {"colour": (0, 200, 200), "text": "Person makes phone call",  "action": "makes phone call", "objects": ["phone"]},
    {"colour": (180, 0, 180), "text": "Person leaves room",       "action": "leaves room",       "objects": ["door"]},
]

FPS = 24
SCENE_DUR = 6          # seconds per scene
TOTAL_DUR = len(SCENES) * SCENE_DUR
WIDTH, HEIGHT = 640, 480
VIDEO_PATH = "synthetic_video.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(VIDEO_PATH, fourcc, FPS, (WIDTH, HEIGHT))

gt_events: List[GroundTruthEvent] = []

for idx, scene in enumerate(SCENES):
    start = idx * SCENE_DUR
    end   = start + SCENE_DUR
    gt_events.append(GroundTruthEvent(
        event_id=idx + 1, actor="Person", action=scene["action"],
        start_time=float(start), end_time=float(end), objects=scene["objects"]
    ))
    for _ in range(FPS * SCENE_DUR):
        frame = np.full((HEIGHT, WIDTH, 3), scene["colour"], dtype=np.uint8)
        cv2.putText(frame, scene["text"], (50, HEIGHT // 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.4, (255, 255, 255), 3)
        writer.write(frame)

writer.release()
print(f"Video saved → {VIDEO_PATH}  ({TOTAL_DUR}s, {FPS} fps, {len(SCENES)} scenes)")
print(f"Ground-truth events: {len(gt_events)}")
for e in gt_events:
    print(f"  [{e.start_time:5.1f}s – {e.end_time:5.1f}s]  {e.actor} {e.action}")

In [ ]:
def sample_frames(video_path: str, sample_fps: float = 1.0) -> List[Tuple[float, np.ndarray]]:
    """Return list of (timestamp_seconds, bgr_frame)."""
    cap = cv2.VideoCapture(video_path)
    native_fps = cap.get(cv2.CAP_PROP_FPS)
    interval = int(round(native_fps / sample_fps))
    frames = []
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % interval == 0:
            ts = idx / native_fps
            frames.append((ts, frame))
        idx += 1
    cap.release()
    return frames

sampled = sample_frames(VIDEO_PATH, sample_fps=1.0)
print(f"Sampled {len(sampled)} frames at 1 fps")

cols = 6
rows = math.ceil(len(sampled) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 2.4))
for i, ax in enumerate(axes.flat):
    if i < len(sampled):
        ts, bgr = sampled[i]
        ax.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{ts:.0f}s", fontsize=9)
    ax.axis("off")
plt.suptitle("Sampled Frames (1 fps)", fontsize=13)
plt.tight_layout()
plt.show()

## Load Vision-Language Model

We load **Qwen2.5-VL-7B-Instruct** in 4-bit quantisation so it fits on a T4 GPU. This model will caption each sampled frame.

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

VLM_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

CACHE_DIR = None
if os.path.isdir("/content/drive/MyDrive"):
    CACHE_DIR = "/content/drive/MyDrive/hf_cache"
    os.makedirs(CACHE_DIR, exist_ok=True)

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

vlm = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_ID, quantization_config=bnb_cfg,
    device_map="auto", cache_dir=CACHE_DIR,
)
vlm_proc = AutoProcessor.from_pretrained(VLM_ID, cache_dir=CACHE_DIR)
print(f"VLM loaded: {VLM_ID}  (4-bit)")

## Frame Captioning

We pass every sampled frame through the VLM with a short prompt asking for a one-sentence description. The resulting list of *(timestamp, caption)* pairs is the foundation of every downstream task — grounding, summarization, and event extraction all build on these captions.

In [ ]:
from qwen_vl_utils import process_vision_info

def caption_frame(pil_img: Image.Image, prompt: str = "Describe this video frame in one sentence.") -> str:
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": pil_img},
            {"type": "text",  "text": prompt},
        ],
    }]
    text = vlm_proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    img_inputs, vid_inputs = process_vision_info(messages)
    inputs = vlm_proc(text=[text], images=img_inputs, videos=vid_inputs,
                      padding=True, return_tensors="pt").to(vlm.device)
    with torch.no_grad():
        ids = vlm.generate(**inputs, max_new_tokens=80)
    out_ids = ids[:, inputs.input_ids.shape[1]:]
    return vlm_proc.batch_decode(out_ids, skip_special_tokens=True)[0].strip()

frame_captions: List[Dict] = []
for ts, bgr in sampled:
    pil = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    cap = caption_frame(pil)
    frame_captions.append({"timestamp": ts, "caption": cap})
    print(f"  {ts:5.1f}s → {cap[:90]}")

df_caps = pd.DataFrame(frame_captions)
df_caps

## Building an Event Timeline

An event timeline groups consecutive frames with similar content into discrete segments. We detect **scene boundaries** by comparing consecutive caption pairs: when the Jaccard word overlap drops below a threshold, we declare a new scene. This simple heuristic works well when scenes are visually distinct. For real-world video, embedding-based similarity would be more robust (see Exercise 1).

In [ ]:
def word_set(text: str) -> set:
    return set(re.findall(r"\w+", text.lower()))

def jaccard(a: str, b: str) -> float:
    sa, sb = word_set(a), word_set(b)
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

@dataclass
class TimelineEvent:
    start_time: float
    end_time: float
    representative_caption: str
    frame_indices: List[int] = field(default_factory=list)

def build_timeline(captions: List[Dict], threshold: float = 0.45) -> List[TimelineEvent]:
    """Segment captions into scenes based on Jaccard similarity."""
    events: List[TimelineEvent] = []
    current = TimelineEvent(
        start_time=captions[0]["timestamp"],
        end_time=captions[0]["timestamp"],
        representative_caption=captions[0]["caption"],
        frame_indices=[0],
    )
    for i in range(1, len(captions)):
        sim = jaccard(captions[i - 1]["caption"], captions[i]["caption"])
        if sim < threshold:
            current.end_time = captions[i - 1]["timestamp"] + 1.0
            events.append(current)
            current = TimelineEvent(
                start_time=captions[i]["timestamp"],
                end_time=captions[i]["timestamp"],
                representative_caption=captions[i]["caption"],
                frame_indices=[i],
            )
        else:
            current.end_time = captions[i]["timestamp"]
            current.frame_indices.append(i)
    current.end_time = captions[-1]["timestamp"] + 1.0
    events.append(current)
    return events

timeline = build_timeline(frame_captions)
print(f"Detected {len(timeline)} scenes:")
for i, ev in enumerate(timeline):
    print(f"  Scene {i+1}: [{ev.start_time:.0f}s – {ev.end_time:.0f}s]  "
          f"({len(ev.frame_indices)} frames)  {ev.representative_caption[:80]}")

In [ ]:
COLOURS = ["#e74c3c", "#2ecc71", "#3498db", "#f1c40f", "#9b59b6",
           "#e67e22", "#1abc9c", "#e84393"]

fig, ax = plt.subplots(figsize=(14, max(3, len(timeline) * 0.7)))
for i, ev in enumerate(timeline):
    duration = ev.end_time - ev.start_time
    ax.barh(i, duration, left=ev.start_time, height=0.6,
            color=COLOURS[i % len(COLOURS)], edgecolor="black", linewidth=0.5)
    label = ev.representative_caption[:50] + ("…" if len(ev.representative_caption) > 50 else "")
    ax.text(ev.start_time + duration / 2, i, label,
            ha="center", va="center", fontsize=8, color="white", fontweight="bold")

ax.set_yticks(range(len(timeline)))
ax.set_yticklabels([f"Scene {i+1}" for i in range(len(timeline))])
ax.set_xlabel("Time (seconds)")
ax.set_title("Detected Event Timeline")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Hierarchical Summarization

Our three-level hierarchy works bottom-up. **Level 1 — Frame captions** are already computed. **Level 2 — Scene summaries** pick the most representative caption from each detected scene. **Level 3 — Video summary** feeds all scene summaries into a text model that produces a coherent paragraph. This separation of concerns keeps each step simple and debuggable.

In [ ]:
scene_summaries = []
for i, ev in enumerate(timeline):
    summary = ev.representative_caption
    scene_summaries.append({
        "scene": i + 1,
        "start": ev.start_time,
        "end":   ev.end_time,
        "summary": summary,
    })

print("Scene-level summaries:")
for s in scene_summaries:
    print(f"  Scene {s['scene']}: [{s['start']:.0f}s–{s['end']:.0f}s]  {s['summary']}")

## Load Text Model for Summarization & Extraction

We load **Qwen3-8B** in 4-bit for text-only tasks — summarization and structured event extraction. A separate text model avoids sending redundant image tokens when we only need language reasoning.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

TEXT_MODEL_ID = "Qwen/Qwen3-8B"

text_tok = AutoTokenizer.from_pretrained(TEXT_MODEL_ID, cache_dir=CACHE_DIR)
text_model = AutoModelForCausalLM.from_pretrained(
    TEXT_MODEL_ID, quantization_config=bnb_cfg,
    device_map="auto", cache_dir=CACHE_DIR,
)
print(f"Text model loaded: {TEXT_MODEL_ID}  (4-bit)")

In [ ]:
def generate_text(prompt: str, max_tokens: int = 256) -> str:
    messages = [{"role": "user", "content": prompt}]
    text = text_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = text_tok(text, return_tensors="pt").to(text_model.device)
    with torch.no_grad():
        ids = text_model.generate(**inputs, max_new_tokens=max_tokens)
    out_ids = ids[:, inputs.input_ids.shape[1]:]
    return text_tok.decode(out_ids[0], skip_special_tokens=True).strip()

scene_block = "\n".join(
    f"- Scene {s['scene']} ({s['start']:.0f}s–{s['end']:.0f}s): {s['summary']}"
    for s in scene_summaries
)

summary_prompt = (
    "Below are scene-level summaries of a 30-second video.\n\n"
    f"{scene_block}\n\n"
    "Write a single coherent paragraph (4-6 sentences) that summarises the "
    "entire video in natural language. Do not use bullet points."
)

video_summary = generate_text(summary_prompt, max_tokens=200)
print("=== Video-Level Summary ===")
print(textwrap.fill(video_summary, width=90))

## Structured Event Extraction

We prompt the text model to convert scene summaries into a JSON array of event records. Each record includes `event_id`, `actor`, `action`, `start_time`, `end_time`, and `objects`. Producing structured JSON lets us load results directly into a DataFrame or database — the same pattern from notebook 07.

In [ ]:
extraction_prompt = (
    "Below are scene descriptions from a video with timestamps.\n\n"
    f"{scene_block}\n\n"
    "Extract a JSON array of events. Each event must have these fields:\n"
    '  event_id (int), actor (string), action (string), '
    'start_time (float), end_time (float), objects (list of strings).\n'
    "Return ONLY the JSON array, no explanation."
)

raw_json = generate_text(extraction_prompt, max_tokens=400)
print("Raw model output:")
print(raw_json[:600])

# Parse JSON — handle model wrapping in markdown code block
json_str = raw_json
if "```" in json_str:
    json_str = json_str.split("```")[1]
    if json_str.startswith("json"):
        json_str = json_str[4:]

try:
    extracted_events = json.loads(json_str)
    df_events = pd.DataFrame(extracted_events)
    print("\n=== Extracted Events ===")
    display(df_events)
except json.JSONDecodeError as e:
    print(f"JSON parse error: {e}")
    print("Using ground-truth events as fallback.")
    extracted_events = [asdict(e) for e in gt_events]
    df_events = pd.DataFrame(extracted_events)
    display(df_events)

## Temporal Grounding — Caption-then-Search

Given a free-text query, we compute **Jaccard word overlap** between the query and every frame caption. The timestamps of the top-scoring frames define the predicted temporal segment. For real-world applications you would replace Jaccard with **embedding cosine similarity** using a sentence-transformer model, which handles paraphrases and synonyms far better (see Exercise 1).

In [ ]:
def ground_query(query: str, captions: List[Dict], top_k: int = 5) -> Tuple[float, float, List[Dict]]:
    """Return (start, end, matched_frames) for the query."""
    scored = []
    for fc in captions:
        sim = jaccard(query, fc["caption"])
        scored.append({**fc, "score": sim})
    scored.sort(key=lambda x: x["score"], reverse=True)
    top = scored[:top_k]
    if not top or top[0]["score"] == 0:
        return (0.0, 0.0, top)
    timestamps = [f["timestamp"] for f in top if f["score"] > 0]
    return (min(timestamps), max(timestamps) + 1.0, top)

QUERIES = [
    ("When does the person pick up an object?",  (6.0, 12.0)),
    ("When does the person leave the room?",     (24.0, 30.0)),
    ("When does the person read a document?",    (12.0, 18.0)),
    ("When does the person enter the room?",     (0.0, 6.0)),
]

grounding_results = []
for query, gt_interval in QUERIES:
    pred_start, pred_end, matches = ground_query(query, frame_captions)
    grounding_results.append({
        "query": query,
        "pred_start": pred_start, "pred_end": pred_end,
        "gt_start": gt_interval[0], "gt_end": gt_interval[1],
    })
    print(f"Q: {query}")
    print(f"   Predicted: [{pred_start:.0f}s – {pred_end:.0f}s]  "
          f"GT: [{gt_interval[0]:.0f}s – {gt_interval[1]:.0f}s]")
    print()

## Grounding Evaluation — Temporal IoU

We measure quality with **temporal IoU**: the intersection of predicted and ground-truth intervals divided by their union. An IoU of 1.0 means perfect alignment.

In [ ]:
def temporal_iou(pred: Tuple[float, float], gt: Tuple[float, float]) -> float:
    inter_start = max(pred[0], gt[0])
    inter_end   = min(pred[1], gt[1])
    intersection = max(0.0, inter_end - inter_start)
    union = (pred[1] - pred[0]) + (gt[1] - gt[0]) - intersection
    return intersection / union if union > 0 else 0.0

rows = []
for r in grounding_results:
    iou = temporal_iou((r["pred_start"], r["pred_end"]),
                       (r["gt_start"],   r["gt_end"]))
    rows.append({"Query": r["query"][:50],
                 "Predicted": f"{r['pred_start']:.0f}–{r['pred_end']:.0f}s",
                 "Ground Truth": f"{r['gt_start']:.0f}–{r['gt_end']:.0f}s",
                 "tIoU": f"{iou:.2f}"})

df_iou = pd.DataFrame(rows)
print("Temporal Grounding Results:")
display(df_iou)
mean_iou = np.mean([temporal_iou((r["pred_start"], r["pred_end"]),
                                 (r["gt_start"], r["gt_end"])) for r in grounding_results])
print(f"\nMean tIoU: {mean_iou:.3f}")

## Clip Retrieval

Clip retrieval returns the actual frames most relevant to a text query. We reuse the grounding similarity scores to find the best-matching timestamps and display the corresponding frames.

In [ ]:
def retrieve_clip(query: str, captions: List[Dict],
                  frames: List[Tuple[float, np.ndarray]], top_k: int = 4):
    """Return top-K frames matching the query."""
    scored = [(jaccard(query, fc["caption"]), i) for i, fc in enumerate(captions)]
    scored.sort(reverse=True)
    indices = [idx for _, idx in scored[:top_k]]
    return [(frames[i][0], frames[i][1]) for i in sorted(indices)]

test_query = "person picks up an object"
clip = retrieve_clip(test_query, frame_captions, sampled, top_k=4)

fig, axes = plt.subplots(1, len(clip), figsize=(14, 3.5))
for ax, (ts, bgr) in zip(axes, clip):
    ax.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{ts:.0f}s")
    ax.axis("off")
fig.suptitle(f'Retrieved clip for: "{test_query}"', fontsize=12)
plt.tight_layout()
plt.show()

## Applications

The three capabilities we built — grounding, summarization, and event extraction — unlock practical applications:

- **Meeting recording analysis:** Generate minutes with action items and decisions linked to timestamps so attendees can jump to the relevant moment.
- **Surveillance & security:** Produce structured event logs from continuous camera feeds, enabling search and alerting without manual review.
- **Content moderation:** Flag specific moments in uploaded videos that may violate platform policies, giving moderators precise timestamps.
- **Sports analytics:** Build highlight reels by extracting goal, foul, and substitution events from full-match footage, then retrieving the corresponding clips.

## Connection to Previous Notebooks

This notebook sits at the intersection of several earlier topics:

- **Notebook 09 — Captioning as a Text Bridge:** Our entire pipeline depends on VLM-generated captions. Caption quality bounds the quality of grounding, summarization, and event extraction.
- **Notebook 07 — Structured Outputs:** Event extraction follows the same JSON-generation pattern — prompting a model for structured data and parsing the result programmatically.
- **Notebook 17 — Video Understanding:** We reuse frame-sampling strategies and extend them with scene detection, temporal grounding, and retrieval.

## Exercises

### Exercise 1 — Embedding-Based Grounding

Replace the Jaccard word-overlap similarity with **sentence embeddings** from a small model such as `sentence-transformers/all-MiniLM-L6-v2`. Encode each frame caption and each query into a dense vector, then rank captions by cosine similarity. Compare the temporal IoU against the word-overlap baseline on the four test queries above. When does embedding-based matching outperform word overlap?

### Exercise 2 — Meeting-Notes Generator

Given the captioned video, prompt the text model to produce structured meeting notes as JSON with three sections: **(a)** a 3-sentence executive summary, **(b)** a list of action items with assignee and deadline, and **(c)** key decisions made. Evaluate by checking whether each field is correctly populated.

### Exercise 3 — Simple Video Search Engine

Build a TF-IDF index over the frame captions using `sklearn.feature_extraction.text.TfidfVectorizer`. Accept free-text queries and return ranked clips with timestamps and thumbnail frames. Measure retrieval quality by checking whether the top-1 result falls within the correct ground-truth interval.

In [ ]:
# === Exercise 1 Starter Code ===
# Uncomment and run after:  pip install sentence-transformers

# from sentence_transformers import SentenceTransformer
# embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
#
# caption_texts = [fc["caption"] for fc in frame_captions]
# caption_embeds = embed_model.encode(caption_texts, normalize_embeddings=True)
#
# def embed_ground(query: str, top_k: int = 5):
#     q_emb = embed_model.encode([query], normalize_embeddings=True)
#     sims = (caption_embeds @ q_emb.T).flatten()
#     top_idx = sims.argsort()[-top_k:][::-1]
#     timestamps = [frame_captions[i]["timestamp"] for i in top_idx]
#     return min(timestamps), max(timestamps) + 1.0
#
# for query, gt in QUERIES:
#     pred = embed_ground(query)
#     iou = temporal_iou(pred, gt)
#     print(f"tIoU={iou:.2f}  {query}")

print("Uncomment the code above after installing sentence-transformers.")

## Key Takeaways

| # | Insight |
|---|---|
| 1 | **Caption-then-search** grounding is simple, modular, and works with any VLM — no task-specific training required. |
| 2 | **Hierarchical summarization** (frame → scene → video) mirrors human cognition and handles redundancy gracefully. |
| 3 | **Scene detection** via caption similarity is a lightweight alternative to visual-feature-based shot boundary detection. |
| 4 | **Structured event extraction** turns video into a queryable database, unlocking downstream automation. |
| 5 | **Temporal IoU** is the standard metric for grounding — it measures overlap between predicted and true time intervals. |
| 6 | Grounding precision is bounded by **sampling rate**: 1 fps means ±1 s resolution at best. Higher fps improves precision but raises cost. |

## References

1. **Moment-DETR** — Lei et al., *"Detecting Moments and Highlights in Videos via Natural Language Queries"* (NeurIPS 2021).
2. **Vid2Seq** — Yang et al., *"Vid2Seq: Large-Scale Pretraining of a Visual Language Model for Dense Video Captioning"* (CVPR 2023).
3. **TimeChat** — Ren et al., *"TimeChat: A Time-sensitive Multimodal Large Language Model for Long Video Understanding"* (CVPR 2024).
4. **Qwen2.5-VL** — Bai et al., *"Qwen2.5-VL Technical Report"* (2025).
5. **Qwen3 Technical Report** — Qwen Team (2025).

In [ ]:
# Clean up synthetic video
if os.path.exists(VIDEO_PATH):
    os.remove(VIDEO_PATH)
    print(f"Removed {VIDEO_PATH}")
print("Notebook complete.")

---

⬅️ [17 — Video Understanding, Frame Sampling & Temporal Reasoning](17_video_understanding_frame_sampling_and_temporal_reasoning.ipynb) · [19 — Serving, Cost, Safety & Evaluation for Multimodal](19_serving_cost_safety_and_evaluation_for_multimodal.ipynb) ➡️